In [2]:
import pandas as pd
import numpy as np
import os

def backtest(average_scores_df, stock):
    average_scores_df = average_scores_df[['datetime', 'instrument', 'score']]
    average_scores_df['datetime'] = average_scores_df['datetime'].astype('datetime64[ns]')
    average_scores_df = average_scores_df.sort_values(by=['datetime', 'instrument'], ascending=[True, True])
    average_scores_df = average_scores_df[average_scores_df['instrument'].isin(stock)]

    stock_df = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
    stock_df['dt'] = pd.to_datetime(stock_df['dt'])
    stock_df = stock_df[['kdcode', 'dt', 'close']]
    stock_df = stock_df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'})
    stock_df = stock_df.sort_values(by=['instrument', 'datetime'])
    stock_df['close_r'] = stock_df['close'] / stock_df['close'].shift(1)
    df_trade = stock_df[stock_df['datetime'] > '2022-12-31']

    x = pd.merge(df_trade, average_scores_df, on=['datetime', 'instrument'], how='outer')
    trade_date_unique = df_trade['datetime'].unique()
    df_return = pd.DataFrame()

    for date in trade_date_unique:
        x_day = x[x['datetime'] == date]
        r_day = x_day.nlargest(10, columns='score').close_r.mean()
        r_day = r_day - 1
        b = {'datetime': date, 'daily_return': r_day}
        df_return = df_return.append(b, ignore_index=True)

    portfolio_df_performance = df_return.set_index(['datetime'])

    alpha_df_performance = pd.DataFrame()
    alpha_df_performance['portfolio_daily_return'] = portfolio_df_performance['daily_return']
    alpha_df_performance['portfolio_net_value'] = (alpha_df_performance['portfolio_daily_return'] + 1).cumprod()

    net_value_columns = ['portfolio_net_value']

    alpha_statistics_df = pd.DataFrame(index=alpha_df_performance[net_value_columns].columns,
                                        columns=["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"])

    alpha_df_performance.index = pd.to_datetime(alpha_df_performance.index)
    alpha_statistics_df.loc[:, "年化收益"] = np.mean((alpha_df_performance[net_value_columns].tail(1)) ** (252 / len(alpha_df_performance)) - 1)
    alpha_statistics_df.loc[:, "年化波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252)
    alpha_statistics_df.loc[:, "最大回撤率"] = np.min((alpha_df_performance[net_value_columns] - alpha_df_performance[net_value_columns].cummax()) / alpha_df_performance[net_value_columns].cummax())
    alpha_statistics_df.loc[:, "夏普率"] = alpha_statistics_df["年化收益"] / alpha_statistics_df["年化波动率"]
    alpha_statistics_df.loc[:, "Calmar"] = alpha_statistics_df["年化收益"] / abs(alpha_statistics_df["最大回撤率"])
    alpha_statistics_df.loc[:, "IR"] = np.mean(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252) / np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)

    return alpha_statistics_df

def get_merged_score_df(base_path, prediction_index):
    score_dfs = []
    prediction_path = os.path.join(base_path, f'prediction_{prediction_index}')
    
    for j in range(20):
        folder_path = os.path.join(prediction_path, str(j))
        all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
        
        for file in all_files:
            df = pd.read_csv(file)
            score_dfs.append(df)
    
    # 合并所有score_dfs并取均值
    merged_df = pd.concat(score_dfs).groupby(['dt', 'kdcode']).mean().reset_index()
    merged_df.rename(columns={'kdcode': 'instrument', 'dt' : 'datetime'}, inplace=True)
    return merged_df

# 主程序
base_path = '/home/liyuante/20240707_csi300_0.8_5_10_ns16'
d = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
stock = d['kdcode'].unique()

results = []

for i in range(20):
    average_scores_df = get_merged_score_df(base_path, i)
    result = backtest(average_scores_df, stock)
    result = result[["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"]]  # 只保留6个指标
    result['folder'] = i
    results.append(result)

# 显示结果
final_df = pd.concat(results).reset_index(drop=True)
print(final_df)


        年化收益     年化波动率     最大回撤率       夏普率    Calmar        IR  folder
0   0.084814  0.194859 -0.134014  0.435260  0.632874  0.586095       0
1   0.537822  0.210215 -0.106638  2.558443  5.043430  2.209404       1
2   0.618282  0.219054 -0.103267  2.822509  5.987223  2.350625       2
3   0.465367  0.205726 -0.115703  2.262069  4.022084  2.054849       3
4   0.360976  0.215146 -0.115794  1.677823  3.117401  1.628509       4
5   0.198524  0.198971 -0.152544  0.997751  1.301422  1.059597       5
6   0.150709  0.201585 -0.109063  0.747617  1.381847  0.887286       6
7   0.343330  0.212751 -0.121929  1.613764  2.815825  1.590171       7
8   0.511836  0.205570 -0.103673  2.489837  4.937030  2.208837       8
9   0.522035  0.237450 -0.124567  2.198504  4.190784  1.976970       9
10  0.108484  0.199923 -0.162536  0.542629  0.667447  0.662616      10
11  0.253681  0.186354 -0.107535  1.361289  2.359059  1.264679      11
12  0.114738  0.207284 -0.131282  0.553533  0.873987  0.674154      12
13  0.

In [2]:
final_df

,年化收益,年化波动率,最大回撤率,夏普率,Calmar,IR,folder
0,0.490968,0.206509,-0.136328,2.377463,3.601368,2.003995,0
1,0.366611,0.220917,-0.108049,1.659495,3.393012,1.490871,1
2,0.272061,0.196657,-0.105733,1.383430,2.573083,1.423053,2
3,0.436592,0.201203,-0.083286,2.169911,5.242065,2.005087,3
4,0.279814,0.215489,-0.169037,1.298504,1.655342,1.347084,4
5,0.198576,0.223120,-0.168900,0.889996,1.175698,1.006126,5
6,0.160267,0.198278,-0.111571,0.808298,1.436456,0.948932,6
7,0.163988,0.202271,-0.110098,0.810735,1.489466,0.942290,7
8,0.559737,0.215636,-0.104841,2.595742,5.338918,2.138518,8
9,0.610354,0.218321,-0.103341,2.795676,5.906200,2.334862,9
